# Prompt Optimization with Tool Calling (Custom Metric, End-to-End Edition)
> **Reference Tutorial:** This notebook accompanies the *Prompt Optimization with Tool Calling* tutorial on SAP Developers.
> Each section below maps directly to a step in the tutorial. Run the cells **in order, top to bottom** — later cells depend on variables created earlier (`client`, `configuration_id`, `execution_id`, etc.).

---

## 👋 Who this notebook is for

You don't need prior experience with SAP AI Core's **Prompt Optimization** service to follow along. You *should* be comfortable with:
- Basic Python (functions, dictionaries, loops)
- Making REST API calls (we use the `requests` library throughout)
- Having a working SAP AI Core / Generative AI Hub tenant with credentials

If any SAP-specific term below is unfamiliar, check the **Glossary** — every step also re-explains the concept in context as it comes up.

## 🧠 What problem are we solving?

Large language models (LLMs) are often asked to look at a user's question and decide **which tools/functions to call** and **with what arguments** ("tool calling" / "function calling"). The quality of the answer depends heavily on how the **system prompt** is written — a vague prompt like `"You are a helpful assistant."` often makes the model respond in plain English instead of structured JSON, which breaks any downstream code expecting machine-readable tool calls.

**Prompt optimization** automates the tedious trial-and-error of prompt engineering. You give SAP AI Core:
1. A **starting ("base") prompt**
2. A **dataset** of example questions and their *correct* tool calls ("golden answers")
3. A **metric** that scores how good a candidate prompt's output is

...and the optimizer uses an LLM to iteratively rewrite the prompt, test it against your dataset, and keep the best-scoring version.

## ⭐ What makes *this* notebook different from a "vanilla" optimization run

This edition goes a step further than simply *using* a pre-existing custom metric — it walks through **creating the custom evaluation metric itself, from inside the notebook**, in addition to running the full optimization pipeline. Concretely, compared to a basic flow, you'll also see:
- A cell that **creates** the `tool-call-accuracy` LLM-as-a-judge metric via a direct API call (rather than assuming it was pre-created in Bruno)
- A combined **configure → trigger → monitor** cell that runs the whole optimization in one go
- A closing comparison between the **custom metric's** optimized prompt and a **built-in `JSON_Match`**-optimized prompt, to see how the choice of metric shapes what the optimizer produces

## 🗺️ Pipeline at a glance

```
 ┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │ 1. Connect to    │ --> │ 2. Verify/create  │ --> │ 3. Configure         │
 │    AI Core       │     │  the custom metric │     │  optimization params │
 └─────────────────┘     └──────────────────┘     └────────────────────┘
                                                              │
                                                              v
 ┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │ 6. Register       │ <-- │ 5. Register base  │ <-- │ 4. Load/normalize +  │
 │  dataset artifact │     │  prompt template   │     │  upload BFCL data     │
 └─────────────────┘     └──────────────────┘     └────────────────────┘
         │
         v
 ┌─────────────────┐     ┌──────────────────┐     ┌────────────────────┐
 │ 7. Configure +    │ --> │ 8. Retrieve the    │ --> │ 9. Compare base vs   │
 │  run + monitor     │     │  optimized prompt   │     │  optimized (live)     │
 └─────────────────┘     └──────────────────┘     └────────────────────┘
```

## 📖 Glossary (SAP AI Core terms used throughout)

| Term | Meaning |
|---|---|
| **AI Core** | SAP's managed platform for running AI workloads (training, inference, orchestration) in the cloud. |
| **Generative AI Hub (`gen_ai_hub`)** | The Python SDK / proxy layer that lets you call LLMs and AI Core services with a unified API, regardless of the underlying model provider. |
| **Resource group** | A logical partition inside an AI Core tenant used to isolate resources between teams or projects. |
| **Scenario** | A named "workflow type" registered in AI Core (e.g. `genai-optimizations`) that groups related artifacts, configurations, and executions together. |
| **Artifact** | A registered reference to a piece of data (here: a folder of dataset files) that AI Core executions can read as input. |
| **Prompt Registry** | A versioned store for prompt templates, referenced by name+version, updatable by the optimizer without you managing raw text files. |
| **Configuration** | A saved combination of parameters (metric, models, dataset filenames, prompt reference, etc.) that fully describes *how* an optimization run should behave — but doesn't run it yet. |
| **Execution** | An actual *run* of a configuration — the long-running job that performs the optimization and produces a result. |
| **Golden record** | One row of your evaluation dataset: an input question plus the *correct* expected output, used to score candidate prompts. |
| **BFCL v3** | The [Berkeley Function-Calling Leaderboard](https://gorilla.cs.berkeley.edu/leaderboard.html) dataset (v3) — a benchmark of questions paired with correct function/tool calls, used here as training/test data. |
| **LLM-as-a-judge** | An evaluation technique where another LLM reads a candidate output and scores its quality against a rubric, instead of (or in addition to) exact string matching. |
| **Evaluation metric (AI Core)** | A registered scoring definition (built-in like `JSON_Match`, or custom like `tool-call-accuracy`) that the optimizer uses to compare candidate prompts. |
| **Orchestration Service** | The AI Core service used to run live inference against a deployed model, optionally chaining prompt templates, grounding, and other modules. |

> ⚠️ **Prerequisites:** Ensure your `.env` file is configured with `AICORE_BASE_URL`, `AICORE_AUTH_URL`, `AICORE_CLIENT_ID`, `AICORE_CLIENT_SECRET`, and `AICORE_RESOURCE_GROUP` before running this notebook. You'll also need the BFCL v3 dataset file (`BFCL_v3_parallel_multiple_10tools.json`) in the same directory.

> ⚠️ **Important gap to know about before you run this notebook:** later in Step 7, this notebook calls a function named `create_config(...)` that is **not defined anywhere in this file** — it's assumed to already exist in your kernel session (e.g. because you ran it earlier from a companion notebook, or defined it yourself). See the callout right before that cell for the full function definition you can paste in if you hit a `NameError`.

---

## Step 1 — Environment Variables Setup & Connect to AI Core

**What this step does:**
Loads credentials from the `.env` file and initializes the `GenAIHubProxyClient` — the main entry point for all AI Core API calls in Python. Every later step re-uses this single `client` object (either directly, or via `client.ai_core_client.base_url` / `client.request_header` to build raw REST calls).

**Why a `.env` file instead of hard-coding credentials?**
Client ID/secret are sensitive. Keeping them in a `.env` file (loaded via `python-dotenv`) means they never get committed to source control or pasted into a notebook that might be shared.

**Create a `.env` file** in the same directory as this notebook with the following content:

```env
AICORE_CLIENT_ID=<your client id>
AICORE_CLIENT_SECRET=<your client secret>
AICORE_AUTH_URL=<your auth url>
AICORE_BASE_URL=<your base url>
AICORE_RESOURCE_GROUP=<your resource group>
```

**Where do these values come from?** They're generated when you create a **service key** for your AI Core service instance in the SAP BTP cockpit — the JSON service key contains `clientid`, `clientsecret`, the OAuth `url` (→ `AICORE_AUTH_URL`), and the API `serviceurls.AI_API_URL` (→ `AICORE_BASE_URL`).

> 💡 No S3 or AWS credentials are required for this flow — all files are uploaded directly to AI Core's built-in dataset storage via the `/lm/dataset/files` endpoint (more on this in Step 4).

In [1]:
from collections import defaultdict
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from dotenv import load_dotenv
import os
import json
import requests
import random
from urllib.parse import quote
from pathlib import Path
from typing import List, Tuple
import time
from ai_api_client_sdk.models.parameter_binding import ParameterBinding
from ai_api_client_sdk.models.input_artifact_binding import InputArtifactBinding
from pydantic import BaseModel
from ai_api_client_sdk.models.artifact import Artifact

load_dotenv(override=True)

# ── SAP AI Core client ────────────────────────────────────────────────────────
client = GenAIHubProxyClient(
    base_url=os.getenv("AICORE_BASE_URL"),
    auth_url=os.getenv("AICORE_AUTH_URL"),
    client_id=os.getenv("AICORE_CLIENT_ID"),
    client_secret=os.getenv("AICORE_CLIENT_SECRET"),
    resource_group=os.getenv("AICORE_RESOURCE_GROUP")
)
resource_group = client.request_header[
    client.ai_core_client.rest_client.resource_group_header
]

print("✅ Connected to AI Core")
print(f"   Resource group: {resource_group}")

✅ Connected to AI Core
   Resource group: grounding


**Reading the output:** `✅ Connected to AI Core` confirms the OAuth handshake succeeded and `client` is ready to use. The **resource group** printed (`grounding` in this run) is the namespace all subsequent artifacts, configurations, prompts, and executions will be created inside.

---

## Custom Metric — Verify, and Create if Missing

**Why a custom metric at all?** The built-in `JSON_Match` metric does a fairly rigid structural/string comparison between the model's JSON output and the golden answer. That's great for catching gross formatting errors, but it can't distinguish "the model called the *wrong* tool" from "the model got the right tool but phrased an argument slightly differently." A **custom LLM-as-a-judge metric** solves this: another LLM reads the candidate output alongside the golden answer and produces a nuanced score based on criteria *you* define (tool name accuracy, argument extraction correctness, valid JSON structure, etc).

This notebook uses `metric=custom` instead of the built-in `JSON_Match`, referencing a metric named `tool-call-accuracy:1.0.0` in the `genai-optimizations` scenario.

**Two-cell pattern used here:**
1. **Verify** — check if `tool-call-accuracy` already exists in your tenant (next cell).
2. **Create** — if it doesn't, define and register it directly from this notebook via a `POST` to `/lm/evaluationMetrics` (the cell after that). This is a slightly more self-contained approach than assuming the metric was pre-created in Bruno/Postman — useful if you're setting this up on a fresh tenant.

> 💡 If the verify cell finds the metric already registered, you don't strictly need to run the create cell — but it's written defensively (handles a `409 Conflict` by looking up the existing ID) so it's safe to run either way.

In [2]:
# ── Verify the custom metric exists before running the pipeline ───────────────
# This cell confirms your custom metric is registered and accessible
url = f"{client.ai_core_client.base_url}/lm/evaluationMetrics"
res = requests.get(url, headers=client.request_header)

if res.status_code != 200:
    print(f"❌ Could not fetch evaluation metrics: {res.status_code} — {res.text}")
else:
    metrics = res.json().get("resources", [])
    print(f"Found {len(metrics)} evaluation metric(s):")
    for m in metrics:
        print(f"  id={m['id']}  name={m['name']}  version={m['version']}  scenario={m.get('scenario', 'N/A')}")

    # Check if our custom metric exists
    found = [m for m in metrics if m["name"] == "tool-call-accuracy"]
    if found:
        print(f"\n✅ Custom metric found: tool-call-accuracy")
        print(f"   id      : {found[0]['id']}")
        print(f"   version : {found[0]['version']}")
        print(f"\n💡 Copy the id above and paste it as CUSTOM_METRIC_ID in Step 2")
    else:
        print("\n❌ Custom metric 'tool-call-accuracy' not found — create it in Bruno first")

Found 31 evaluation metric(s):
  id=7be25dc0-0cc9-4d84-bf87-47cf30cef56f  name=tool-call-accuracy-numeric  version=1.0.0  scenario=genai-optimizations
  id=721b9784-df86-4fb3-93e8-f46828d02a03  name=tool-call-accuracy  version=1.0.0  scenario=genai-optimizations
  id=6867b1a9-4572-4b1c-b30e-340104847ca5  name=test-metrics  version=0.0.1  scenario=genai-evaluations-test
  id=a39ee422-2a5d-4d3a-9518-728e3aa09975  name=test-metric  version=0.0.1  scenario=genai-evaluations-test
  id=9b349f8e-cd39-486d-809c-3dcfa3b15ac7  name=groundedness  version=0.1.6  scenario=genai-evaluations
  id=2b3cc135-a031-4d93-8641-1f3833797034  name=groundedness  version=0.0.1  scenario=genai-evaluations
  id=93a16045-d577-4132-8481-9497cb205961  name=BERT Score  version=1.0.0  scenario=genai-evaluations
  id=3ea07c1f-5b10-4b12-bf46-6d429faf8010  name=BLEU  version=1.0.0  scenario=genai-evaluations
  id=3904208a-b886-41b1-8448-d363245d5397  name=ROUGE  version=1.0.0  scenario=genai-evaluations
  id=21a84cc7-7fc

**Reading the output:** this lists every evaluation metric registered in your tenant (built-in ones like `groundedness`, `BLEU`, `ROUGE`, `JSON_Match`, `Exact Match`, plus any custom ones), then filters for `tool-call-accuracy`. In this run it was **already found** (`id=562ddf0b-...`) — meaning the create cell below isn't strictly required this time, but it's still safe to run (see note above).

### Creating the metric definition

This cell defines the **rubric** the judge LLM will use to score candidate prompts, then registers it. A few pieces worth understanding:

- **`evaluationMethod: "llm-as-a-judge"`** — tells AI Core this metric isn't a simple string comparison; it invokes an LLM (configured under `spec.configuration.modelConfiguration`, here `gemini-2.5-pro:001`) to produce a rating.
- **`ratingRubric`** — a discrete 1/3/5 scale with plain-English rules for each rating band (5 = perfect JSON + correct tools + correct args, 3 = right tools but imperfect args, 1 = invalid JSON or wrong tools entirely). This is what keeps the judge's scoring consistent across thousands of candidate evaluations.
- **`evaluationSteps`** — a short checklist the judge LLM is instructed to follow (valid JSON? correct top-level tool names? correct arguments?) before assigning a rating — this reduces variance in how the judge reasons about each response.
- **`examples`** — a few-shot example embedded directly in the metric definition, showing the judge exactly what a rating-5 response looks like.
- The `if res.status_code == 201` / `elif == 409` branch means this cell is **idempotent**: run it once to create the metric, or run it again later and it'll just fetch the existing ID instead of erroring out.

In [3]:
# ── Create custom metric with correct usageType ───────────────────────────────
url = f"{client.ai_core_client.base_url}/lm/evaluationMetrics"

custom_metric_body = {
    "scenario": "genai-optimizations",
    "name": "tool-call-accuracy",
    "version": "1.0.0",
    "description": "Measures how accurately the model identifies and structures the correct tool calls",
    "evaluationMethod": "llm-as-a-judge",
    "metricType": "optimization",
    "usageType": ["optimization"],
    "includeProperties": ["prompt", "reference"],
    "spec": {
        "promptType": "structured",
        "configuration": {
            "modelConfiguration": {
                "name": "gemini-2.5-pro",
                "version": "001"
            },
            "promptConfiguration": {
                "definition": "Measures how well the model output matches the expected tool call JSON structure",
                "evaluationTask": "Rate how accurately the response identifies and structures the correct tool calls compared to the reference",
                "ratingRubric": [
                    {
                        "rating": 5,
                        "rule": "Response is a valid JSON object with all correct tool names and arguments exactly matching the reference"
                    },
                    {
                        "rating": 3,
                        "rule": "Response has correct tool names but some arguments are missing or slightly incorrect"
                    },
                    {
                        "rating": 1,
                        "rule": "Response is not valid JSON or has completely wrong tool names and arguments"
                    }
                ],
                "criteria": "Evaluate based on correct tool name identification, argument extraction accuracy, and valid JSON structure",
                "evaluationSteps": [
                    "Check if the response is a valid JSON object",
                    "Verify the top-level keys match the expected tool names from the reference",
                    "Check each tool's arguments match the expected values",
                    "Rate based on overall accuracy of tool call extraction"
                ],
                "examples": [
                    {
                        "prompt": "What is the weather in Tokyo for the next 3 days in celsius?",
                        "groundingInput": "",
                        "groundingOutput": "",
                        "response": "{\"weather_forecast\": {\"location\": [\"Tokyo\"], \"days\": [3], \"units\": [\"celsius\"]}}",
                        "reference": "{\"weather_forecast\": {\"location\": [\"Tokyo\"], \"days\": [3], \"units\": [\"celsius\"]}}",
                        "rating": 5,
                        "explanation": "Response is a perfectly valid JSON with correct tool name and all arguments matching the reference exactly"
                    }
                ]
            }
        }
    }
}

res = requests.post(
    url,
    headers={**client.request_header, "Content-Type": "application/json"},
    json=custom_metric_body
)

print(f"Status: {res.status_code}")
print(res.json())

if res.status_code == 201:
    CUSTOM_METRIC_ID = res.json()["id"]
    print(f"\n✅ Custom metric created successfully")
    print(f"   id      : {CUSTOM_METRIC_ID}")
    print(f"   name    : tool-call-accuracy")
    print(f"   version : 1.0.0")
    print(f"\n💡 CUSTOM_METRIC_ID has been set automatically — proceed to next cell")
elif res.status_code == 409:
    print("\n⚠️  Metric already exists — fetching existing ID...")
    list_res = requests.get(url, headers=client.request_header)
    for m in list_res.json().get("resources", []):
        if m["name"] == "tool-call-accuracy":
            CUSTOM_METRIC_ID = m["id"]
            print(f"   ✅ Found existing metric id: {CUSTOM_METRIC_ID}")
            break
else:
    print(f"\n❌ Failed to create metric: {res.text}")

Status: 201
{'id': 'bdb69976-034f-45fe-98fd-dd5534766824', 'name': 'tool-call-accuracy', 'version': '1.0.0', 'scenario': 'genai-optimizations', 'createdAt': '2026-07-01 17:14:53.421238', 'message': 'optimization metric successfully created'}

✅ Custom metric created successfully
   id      : bdb69976-034f-45fe-98fd-dd5534766824
   name    : tool-call-accuracy
   version : 1.0.0

💡 CUSTOM_METRIC_ID has been set automatically — proceed to next cell


**Reading the output:** `Status: 201` confirms a brand-new metric was created, and its `id` is printed both in the raw JSON response and again in the friendly `✅ Custom metric created successfully` summary. **Copy this `id`** — you'll need it as `CUSTOM_METRIC_ID` in the next step. (If you re-run this cell after the metric already exists, you'd instead see the `409` branch handle it gracefully and reuse the existing ID.)

---

## Step 2 — Configure Optimization Parameters

**What this step does:**
Defines all configuration constants used throughout the notebook — dataset path, prompt name/version, reference model, target model, metric, and the Pydantic models for the prompt template spec.

**"Reference" vs "target" model — what's the difference?**
- The **reference model** (`REFERENCE_MODEL`, e.g. `gpt-4o:2024-08-06`) acts as a *teacher* the optimizer can compare against while it searches for a better prompt.
- The **target model(s)** (`TARGET_MODELS`) are the model(s) the final optimized prompt is actually being tuned *for*. Different models respond differently to identical prompt wording, so a prompt optimized for Gemini 2.5 Pro may look different from one optimized for GPT-4o.

**Why split into train/test samples?**
- **Train samples** (`N_TRAIN_SAMPLES = 25`) are what the optimizer actively uses to *generate and refine* candidate prompts.
- **Test samples** (`N_TEST_SAMPLES = 15`) are held out and only used to *score* each candidate prompt, so the reported score reflects genuine generalization rather than memorization — the same train/test split idea used in traditional ML.

**Key parameters:**
| Parameter | Value | Description |
|---|---|---|
| `REFERENCE_MODEL` | `gpt-4o:2024-08-06` | Teacher model used for evaluation |
| `TARGET_MODELS` | `gemini-2.5-pro:001` | Model to optimize the prompt for |
| `METRIC` | `custom` | Uses LLM-as-a-judge custom metric `tool-call-accuracy:1.0.0` |
| `CUSTOM_METRIC_ID` | *(hardcoded below)* | ID of the metric verified/created in the previous step |
| `N_TRAIN_SAMPLES` | 25 | Number of samples used to train/refine the prompt |
| `N_TEST_SAMPLES` | 15 | Number of samples used to evaluate candidate prompts |

**What are the Pydantic models for?**
`PromptTemplateMsg` and `PromptTemplateSpec` give the prompt template a strict, typed shape (a list of `{role, content}` messages) before it's serialized to JSON and pushed to the Prompt Registry in Step 5. Using Pydantic here catches typos/shape errors locally instead of getting a cryptic 400 error from the API.

> ⚠️ Verify that `gpt-4o:2024-08-06` and `gemini-2.5-pro:001` are available in your AI Core tenant before running. Check via Generative AI Hub → Models.

> 💡 Note: `CUSTOM_METRIC_ID` is hard-coded in the cell below to a specific ID string. If you're re-running this on your own tenant, replace it with the `id` your verify/create cells printed above — otherwise the optimizer will fail to find the metric.

In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────
BFCL_DATASET      = "BFCL_v3_parallel_multiple_10tools.json"
BFCL_DATASET_MODE = "parallel_multiple"
N_TRAIN_SAMPLES   = 25
N_TEST_SAMPLES    = 15

PROMPT_NAME    = "bfcl-tool-base"
PROMPT_VERSION = "0.0.1"
SCENARIO       = "genai-optimizations"

SYSTEM_PROMPT   = "You are a helpful assistant."
PROMPT_TEMPLATE = "{{?question}}"
FIELDS          = ["question"]

# Use a model confirmed available in your region
REFERENCE_MODEL = "gpt-4o:2024-08-06"
TARGET_MODELS = {
    "gemini-2.5-pro:001": "bfcl-tool-optimized-custom:0.0.1",
}
# Custom metric ID from Bruno — tool-call-accuracy:1.0.0
CUSTOM_METRIC_ID = "4045e80b-939c-4fea-8de5-a0a00b7f7bc3"  # paste the id returned from Bruno here
METRIC = "custom"

# ── Pydantic models ───────────────────────────────────────────────────────────
class PromptTemplateMsg(BaseModel):
    role: str
    content: str

class PromptTemplateSpec(BaseModel):
    template: List[PromptTemplateMsg]

prompt = PromptTemplateSpec(template=[
    PromptTemplateMsg(role="system", content=SYSTEM_PROMPT),
    PromptTemplateMsg(role="user",   content=PROMPT_TEMPLATE),
])

print("✅ Configuration set")
print(f"   Dataset       : {BFCL_DATASET}")
print(f"   Scenario      : {SCENARIO}")
print(f"   Prompt name   : {PROMPT_NAME}:{PROMPT_VERSION}")
print(f"   Reference     : {REFERENCE_MODEL}")
print(f"   Target        : {list(TARGET_MODELS.keys())}")
print(f"   Metric        : {METRIC} (custom metric: tool-call-accuracy_numeric:1.0.0)")

✅ Configuration set
   Dataset       : BFCL_v3_parallel_multiple_10tools.json
   Scenario      : genai-optimizations
   Prompt name   : bfcl-tool-base:0.0.1
   Reference     : gpt-4o:2024-08-06
   Target        : ['gemini-2.5-pro:001']
   Metric        : custom (custom metric: tool-call-accuracy_numeric:1.0.0)


**Reading the output:** this echoes back the configuration you set, so you can sanity-check it before anything gets created remotely.

---

## Step 3 — Load and Normalize the BFCL v3 Dataset

**What this step does:**
Loads the BFCL v3 dataset file and normalizes it into the SAP optimizer "golden format." The optimizer doesn't understand BFCL's native structure — it expects each example as a **golden record**: `{"fields": {"question": ...}, "answer": "<JSON string>"}`. So this step is a translation layer between "the format the benchmark ships in" and "the format the optimizer needs."

Several helper functions work together to do this translation. We've split the dense original code into smaller, labeled pieces below so each piece's job is clear:

1. **`read_bfcl_file`** — a robust reader that handles 3 BFCL file formats: JSON array, standard JSONL, and concatenated JSON objects (the native BFCL v3 format).
2. **`normalize_bfcl_tool`** — converts raw BFCL tool definitions to OpenAI ChatCompletions format (e.g., `float` → `number`, `dict` → `object`, `any` → `string`).
3. **`dedupe_tool_name`** / **`union_bfcl_tools`** — deduplicates tools with identical names but different schemas across samples.
4. **`detect_tool_key`** / **`detect_question_key`** — auto-detects the correct field names in the dataset.
5. **`build_golden`** — converts each BFCL sample into a SAP optimizer golden record:
   - `fields.question` → the user query text
   - `answer` → a JSON object string of merged tool calls (e.g. `{"weather_forecast": {...}, "calculate_distance": {...}}`)

**Output format per golden record:**
```json
{
  "fields": { "question": "I'm planning a trip to Japan..." },
  "answer": "{\"currency_conversion\": {\"amount\": [5000.0], ...}, \"calculate_distance\": {...}}"
}
```

> 💡 The `answer` must be a JSON **object** string (not an array), where each key is a tool name and each value is its arguments dict.

### 3.1 — Robustly reading the raw BFCL file

BFCL v3's native file format isn't a single JSON array or clean JSONL — it's a stream of back-to-back JSON objects with no separators (`{...}{...}{...}`), which trips up a plain `json.load()`. `read_bfcl_file` tries three strategies in order and falls back gracefully:

1. Is the whole file one JSON array `[ {...}, {...} ]`? Parse it directly (and un-wrap double-encoded strings if needed).
2. Is it standard JSONL (one JSON object per line)? Parse line-by-line.
3. Otherwise, assume it's concatenated JSON objects and use Python's `json.JSONDecoder().raw_decode()` in a loop to scan through the text character-by-character, pulling out one valid object at a time — this is what actually handles native BFCL v3 files.

In [5]:
# ── BFCL v3 file reader ───────────────────────────────────────────────────────
def read_bfcl_file(file_path: Path) -> list:
    """Robust reader for BFCL v3 files (concatenated JSON objects)."""
    with open(file_path, "r") as f:
        content = f.read().strip()

    print(f"File size: {len(content):,} bytes")

    # Format 1: JSON array [ {...}, {...} ]
    if content.startswith("["):
        try:
            result = json.loads(content)
            if result and isinstance(result[0], str):
                print("Detected double-encoded strings — decoding...")
                result = [json.loads(item) for item in result]
            print(f"Loaded as JSON array: {len(result)} records")
            return result
        except json.JSONDecodeError:
            pass

    # Format 2: Standard JSONL — one complete object per line
    if "\n" in content:
        objects = []
        for line in content.split("\n"):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict):
                    objects.append(obj)
            except json.JSONDecodeError:
                pass
        if objects:
            print(f"Loaded as JSONL: {len(objects)} records")
            return objects

    # Format 3: Concatenated/multi-line JSON objects — use raw_decode to scan through
    # This is the correct format for BFCL v3: {...}\n{...}\n{...}
    objects = []
    decoder = json.JSONDecoder()
    idx = 0
    while idx < len(content):
        while idx < len(content) and content[idx] in " \t\n\r":
            idx += 1
        if idx >= len(content):
            break
        if content[idx] != "{":
            idx += 1
            continue
        try:
            obj, end_idx = decoder.raw_decode(content, idx)
            if isinstance(obj, dict):
                objects.append(obj)
            idx = end_idx
        except json.JSONDecodeError:
            idx += 1
            continue

    print(f"Loaded as concatenated JSON: {len(objects)} records")
    return objects


### 3.2 — Normalizing tool/function schemas

BFCL tool definitions use its own type vocabulary (`float`, `dict`, `any`) that doesn't match the OpenAI-style function-calling schema most LLM proxies expect (`number`, `object`, `string`). `normalize_bfcl_tool` walks each tool's parameter definitions and remaps these types, and also sanitizes tool names (`reformat_tool_name` replaces dots with underscores, since dotted names aren't valid function identifiers for most providers). The result is wrapped in the `{"type": "function", "function": {...}}` envelope that most chat-completion APIs expect.

In [6]:
# ── BFCL v3 normalisation ─────────────────────────────────────────────────────
def reformat_tool_name(name: str) -> str:
    return name.replace(".", "_")

def normalize_bfcl_tool(tool: dict) -> dict:
    """Convert raw BFCL tool definition to OpenAI ChatCompletions format."""
    tool = json.loads(json.dumps(tool))       # deep copy
    tool["parameters"]["type"] = "object"     # BFCL uses 'dict'
    tool["name"] = reformat_tool_name(tool["name"])
    for param in tool["parameters"].get("properties", {}).values():
        if param["type"] == "float":
            param["type"] = "number"
        elif param["type"] == "dict":
            param["type"] = "object"
        elif param["type"] == "any":
            param["type"] = "string"
        elif param["type"] == "array" and "items" in param:
            if param["items"].get("type") == "float":
                param["items"]["type"] = "number"
            elif param["items"].get("type") == "dict":
                param["items"]["type"] = "object"
    return {"type": "function", "function": tool}



### 3.3 — Deduplicating tools and detecting field names across samples

A single dataset file can define the *same-named* tool slightly differently across different samples. `union_bfcl_tools` walks every sample, and if it finds a tool name reused with a genuinely different schema, it renames the newer one (`dedupe_tool_name` appends a suffix like `_n2`) so both variants can coexist in the unioned tool list without silently overwriting each other — and it prints a `WARNING` whenever this happens, which is useful for auditing data quality.

`detect_tool_key` and `detect_question_key` are small defensive helpers: instead of hard-coding `sample["function"]` and `sample["question"]`, they check a short list of plausible key names first. This makes the loader resilient if you swap in a differently-named BFCL variant later.

In [7]:
def dedupe_tool_name(tool_name: str, answers: list, occurrences: int):
    new_name = f"{tool_name}_n{occurrences}"
    new_answers = []
    for answer in answers:
        new_answer = {}
        for k, v in answer.items():
            new_answer[reformat_tool_name(new_name if k == tool_name else k)] = v
        new_answers.append(new_answer)
    return new_name, new_answers

def union_bfcl_tools(samples: list, tool_key: str = "function") -> list:
    """Union and deduplicate all tools across samples into one normalized list."""
    tool_map = {}
    tool_name_occurrences = defaultdict(int)
    tool_name_to_definition = {}
    for sample in samples:
        for tool in sample[tool_key]:
            tool_name = tool["name"]
            tool_name_occurrences[tool_name] += 1
            is_duplicate = (
                tool_name in tool_name_to_definition and
                tool != tool_name_to_definition[tool_name]
            )
            if is_duplicate:
                new_name, new_answers = dedupe_tool_name(
                    tool_name, sample["answer"], tool_name_occurrences[tool_name]
                )
                print(f"WARNING: duplicate tool {tool_name!r} → renamed to {new_name!r}")
                tool["name"]     = new_name
                sample["answer"] = new_answers
            tool_name_to_definition[tool["name"]] = tool
            normalized = normalize_bfcl_tool(tool)
            tool_map[normalized["function"]["name"]] = normalized
    return list(tool_map.values())

def detect_tool_key(sample: dict) -> str:
    """Detect which key holds the tool definitions in a BFCL sample."""
    for candidate in ["function", "functions", "tools", "tool"]:
        if candidate in sample:
            return candidate
    raise KeyError(
        f"Cannot find tool key in sample. Available keys: {list(sample.keys())}"
    )

def detect_question_key(sample: dict) -> str:
    """Detect which key holds the question/messages in a BFCL sample."""
    for candidate in ["question", "messages", "turns", "prompt"]:
        if candidate in sample:
            return candidate
    raise KeyError(
        f"Cannot find question key in sample. Available keys: {list(sample.keys())}"
    )



### 3.4 — Building golden records and running the full load

`build_golden` takes one raw BFCL sample and produces exactly the `{"fields": ..., "answer": ...}` shape the optimizer needs:
- It joins multi-turn question text into a single string.
- It merges the (potentially multiple) tool-call dicts BFCL provides per sample into one flat `merged_answer` dict, keyed by tool name.
- It flattens a BFCL quirk where some values arrive as a nested single-element list, and stringifies floats so they match the normalized string-typed schema.

`load_bfcl_dataset` ties everything together: read the file → randomly sample `n_train + n_test` records (with a fixed `random.seed(42)` for reproducibility) → detect field names → union/normalize the tools → build golden records for the train and test splits.

**Reading the output below:** you should see the file size, which format was auto-detected, the detected key names, the resulting train/test counts, how many *unique* normalized tools were unioned, and finally a full example of one golden record — this is your chance to visually sanity-check that the question and answer look correct before uploading anything.

In [8]:
def build_golden(sample: dict, question_key: str = "question") -> dict:
    """Convert one BFCL sample to a SAP optimizer golden record."""
    raw_question = sample[question_key]
    if isinstance(raw_question[0], list):
        question = "\n".join(q["content"] for q in raw_question[0])
    elif isinstance(raw_question[0], dict):
        question = "\n".join(q["content"] for q in raw_question)
    else:
        question = str(raw_question)

    # Merge all tool calls into a single flat dict
    # e.g. [{"weather": {...}}, {"hotel": {...}}] → {"weather": {...}, "hotel": {...}}
    merged_answer = {}
    for answer in sample["answer"]:
        for key, value in answer.items():
            key = reformat_tool_name(key)
            # Flatten nested single-element arrays (BFCL quirk)
            if isinstance(value, list) and len(value) == 1 and isinstance(value[0], list):
                value = value[0]
            # Float → string to match normalized tool schema
            if isinstance(value, float):
                value = str(value)
            elif isinstance(value, list):
                value = [str(v) if isinstance(v, float) else v for v in value]
            merged_answer[key] = value

    # answer must be a JSON object string, not a JSON array string
    return {
        "fields": {"question": question},
        "answer": json.dumps(merged_answer)   # "{...}" not "[{...}]"
    }

def load_bfcl_dataset(
    dataset_path: Path,
    n_train: int,
    n_test: int,
) -> Tuple[list, list, list]:
    """Load, sample, normalise, and split BFCL v3 data."""
    samples = read_bfcl_file(dataset_path)
    print(f"Total records in file: {len(samples)}")

    if samples:
        print(f"Sample keys: {list(samples[0].keys())}")

    n_total = min(n_train + n_test, len(samples))
    random.seed(42)
    sampled = random.sample(samples, n_total)

    tool_key     = detect_tool_key(sampled[0])
    question_key = detect_question_key(sampled[0])
    print(f"Tool key: '{tool_key}' | Question key: '{question_key}'")

    tools         = union_bfcl_tools(sampled, tool_key=tool_key)
    train_goldens = [build_golden(s, question_key=question_key) for s in sampled[:n_train]]
    test_goldens  = [build_golden(s, question_key=question_key) for s in sampled[n_train:]]
    return train_goldens, test_goldens, tools

# ── Load dataset ──────────────────────────────────────────────────────────────
dataset_path = Path(BFCL_DATASET)
train_goldens, test_goldens, tools = load_bfcl_dataset(
    dataset_path, N_TRAIN_SAMPLES, N_TEST_SAMPLES
)
print(f"Train goldens : {len(train_goldens)}")
print(f"Test goldens  : {len(test_goldens)}")
print(f"Unioned tools : {len(tools)}")
print(f"\nSample golden:\n{json.dumps(train_goldens[0], indent=2)}")

File size: 428,329 bytes
Loaded as concatenated JSON: 35 records
Total records in file: 35
Sample keys: ['id', 'question', 'function', 'answer']
Tool key: 'function' | Question key: 'question'
Train goldens : 25
Test goldens  : 10
Unioned tools : 10

Sample golden:
{
  "fields": {
    "question": "I'm planning a trip to Japan. I have 5000 US dollars and want to know how much that is in Japanese Yen. I'd also like to know the distance from Tokyo to Kyoto in kilometers. And while I'm researching Japanese companies, could you get me the latest stock price for Toyota?"
  },
  "answer": "{\"currency_conversion\": {\"amount\": [5000.0], \"from_currency\": [\"USD\", \"US Dollars\", \"US Dollar\"], \"to_currency\": [\"JPY\", \"Japanese Yen\"]}, \"calculate_distance\": {\"origin\": [\"Tokyo\"], \"destination\": [\"Kyoto\"], \"unit\": [\"km\", \"\"]}, \"get_stock_info\": {\"company\": [\"Toyota\", \"TM\"], \"metric\": [\"price\"]}}"
}


---

## Step 4 — Upload Dataset Files to AI Core Storage

**What this step does:**
Serializes the four prepared objects to local JSON files, then uploads each one to a shared folder in AI Core's built-in dataset storage using the `/lm/dataset/files` endpoint.

**Files uploaded:**
| File | Contents |
|---|---|
| `bfcl_train.json` | 25 golden records used to train/refine the prompt |
| `bfcl_test.json` | 15 golden records used to evaluate candidate prompts |
| `bfcl_tools.json` | Union of all normalized tool definitions across samples |
| `bfcl_prompt_template.json` | The base prompt template spec |

All four files land in the same remote folder: `default/datasets/bfcl-optimizer/`

After uploading, the shared folder is registered as a single **dataset artifact** under the `genai-optimizations` scenario. The optimizer reads all files from this artifact folder.

> 💡 `get_or_create_artifact` checks for an existing artifact at the same URL before creating a new one — safe to re-run without creating duplicates.

In [9]:
# ── Serialize files for upload ────────────────────────────────────────────────
train_local  = "./bfcl_train.json"
test_local   = "./bfcl_test.json"
tools_local  = "./bfcl_tools.json"
prompt_local = "./bfcl_prompt_template.json"

with open(train_local,  "w") as f: json.dump(train_goldens,    f, indent=2)
with open(test_local,   "w") as f: json.dump(test_goldens,     f, indent=2)
with open(tools_local,  "w") as f: json.dump(tools,            f, indent=2)
with open(prompt_local, "w") as f: json.dump(prompt.model_dump(), f, indent=2)
print("Local files written.")



Local files written.


### 4.1 — The upload helper

`upload_file` does a raw HTTP `PUT` to AI Core's dataset-file endpoint. A few details worth noting:
- The remote path is URL-encoded (`quote(full_path, safe="")`) because it will contain slashes that need to survive being embedded in a URL path segment.
- `params={"overwrite": "true"}` means re-running this cell won't error out if the files already exist — it just replaces them, which is handy while you're iterating on the dataset.
- It returns the *folder* path (not the individual file path), since that's what gets registered as a single artifact in the next cell.

In [10]:
# ── Helpers: upload & artifact ────────────────────────────────────────────────
def upload_file(local_path: str, remote_subfolder: str, filename: str) -> str:
    """Upload file to AI Core dataset storage. Returns folder path 'default/<subfolder>'."""
    full_path    = f"default/{remote_subfolder}/{filename}"
    encoded_path = quote(full_path, safe="")
    url          = f"{client.ai_core_client.base_url}/lm/dataset/files/{encoded_path}"
    headers      = {**client.request_header, "Content-Type": "application/json"}
    with open(local_path, "rb") as f:
        res = requests.put(url, params={"overwrite": "true"}, headers=headers, data=f)
    print(f"  Upload [{filename}]: {res.status_code}")
    res.raise_for_status()
    return f"default/{remote_subfolder}"



### 4.2 — Uploading all four files and registering the artifact

This cell does two distinct things:

1. **Uploads** each of the four local files into the shared remote folder `default/datasets/bfcl-optimizer/`.
2. **Registers that folder as an artifact** via `get_or_create_artifact`. An artifact is AI Core's way of turning "a folder of files sitting in storage" into a first-class, referenceable object (with an `artifact_id`) that a configuration can bind as an input. The `ai://` prefix is AI Core's internal storage scheme.

`get_or_create_artifact` first **queries existing artifacts** in the scenario and reuses one if the same URL is already registered — that's why you may see `Reusing artifact [...]` printed instead of `Created artifact [...]` on repeated runs. Either way, hang onto the printed **Artifact ID** — it's used to build the optimization configuration in Step 6.

In [11]:
def get_or_create_artifact(name: str, folder_path: str, description: str) -> str:
    """Register folder as artifact. Returns artifact_id."""
    artifact_url = f"ai://{folder_path}"
    existing = client.ai_core_client.artifact.query(
        resource_group=resource_group, scenario_id=SCENARIO
    )
    for art in existing.resources:
        if art.url == artifact_url:
            print(f"  Reusing artifact [{name}]: {art.id}")
            return art.id
    resp = client.ai_core_client.artifact.create(
        name=name, kind=Artifact.Kind.DATASET,
        url=artifact_url, scenario_id=SCENARIO,
        resource_group=resource_group, description=description
    )
    print(f"  Created artifact [{name}]: {resp.id}")
    return resp.id

# ── Upload all files to shared folder ────────────────────────────────────────
REMOTE_SUBFOLDER = "datasets/bfcl-optimizer"
print("Uploading files...")
shared_folder = upload_file(train_local,  REMOTE_SUBFOLDER, "bfcl_train.json")
upload_file(test_local,   REMOTE_SUBFOLDER, "bfcl_test.json")
upload_file(tools_local,  REMOTE_SUBFOLDER, "bfcl_tools.json")
upload_file(prompt_local, REMOTE_SUBFOLDER, "bfcl_prompt_template.json")
print(f"Shared folder: {shared_folder}")

# ── Register dataset artifact ─────────────────────────────────────────────────
optimizer_artifact_id = get_or_create_artifact(
    name="bfcl-optimizer-data",
    folder_path=shared_folder,
    description="BFCL train/test goldens, tools, and prompt template"
)
print(f"Artifact ID: {optimizer_artifact_id}")


Uploading files...
  Upload [bfcl_train.json]: 201
  Upload [bfcl_test.json]: 201
  Upload [bfcl_tools.json]: 201
  Upload [bfcl_prompt_template.json]: 201
Shared folder: default/datasets/bfcl-optimizer
  Reusing artifact [bfcl-optimizer-data]: e4a97767-8130-4e8c-9ded-b0313eb7d4ad
Artifact ID: e4a97767-8130-4e8c-9ded-b0313eb7d4ad


---

## Step 5 — Create and Register the Base Prompt Template

**What this step does:**
Pushes the base prompt template to the Prompt Registry under the `genai-optimizations` scenario.

The base prompt is intentionally minimal:
- **System:** `"You are a helpful assistant."`
- **User:** `{{?question}}`

**Why start so minimal?** The whole point of prompt optimization is to let the optimizer discover a *better* prompt than you'd write by hand — starting from a deliberately weak, generic prompt gives the optimizer maximum room to add structure (JSON-only output instructions, tool schemas, reasoning steps, etc.) and lets you clearly see, in Step 9, how much value the optimization actually added.

The optimizer takes this as its starting point and iteratively rewrites it during execution. When done, the final refined prompt is saved back to the registry under the name specified in `targetPromptMapping`.

**What does `{{?question}}` mean?** It's a template placeholder — at inference time, the actual user question gets substituted in for `{{?question}}`. You'll see this same substitution pattern used later in `run_inference` (`user_content = user_template.replace("{{?question}}", question)`).

> 💡 A `409` response means the prompt already exists — this is safe and the existing version is reused automatically.

In [12]:
# ── Push prompt to registry ───────────────────────────────────────────────────
def push_prompt(spec: PromptTemplateSpec, name: str, version: str, scenario: str):
    url  = f"{client.ai_core_client.base_url}/lm/promptTemplates"
    body = {"name": name, "version": version, "scenario": scenario, "spec": spec.model_dump()}
    res  = requests.post(
        url,
        headers={**client.request_header, "Content-Type": "application/json"},
        json=body
    )
    print(f"Prompt registry: {res.status_code} — {res.json().get('message', '')}")
    if res.status_code == 409:
        print("Prompt already exists — reusing.")
        return {"name": name, "version": version}
    res.raise_for_status()
    return res.json()

push_prompt(prompt, PROMPT_NAME, PROMPT_VERSION, SCENARIO)

Prompt registry: 200 — Prompt updated successfully.


{'message': 'Prompt updated successfully.',
 'id': '8221e74e-cb20-4fb1-99eb-2092fff0d77b',
 'scenario': 'genai-optimizations',
 'name': 'bfcl-tool-base',
 'version': '0.0.1'}

In [16]:
## Step 6 & 7 — Register a Configuration, Trigger the Execution, and Monitor It


CUSTOM_METRIC_ID = "7be25dc0-0cc9-4d84-bf87-47cf30cef56f"

# ── Create configuration using the new numerical custom metric ────────────────
def create_config(
    metric: str,
    reference_model: str,
    targets: dict,
    train_filename: str,
    test_filename: str,
    prompt_artifact_id: str,
    prompt_name: str,
    prompt_version: str,
    scenario: str,
) -> str:
    base_prompt = f"{scenario}/{prompt_name}:{prompt_version}"

    input_parameters = [
        ParameterBinding(key="basePrompt",             value=base_prompt),
        ParameterBinding(key="baseModel",              value=reference_model),
        ParameterBinding(key="targetModels",           value=",".join(targets.keys())),
        ParameterBinding(
            key="targetPromptMapping",
            value=",".join(f"{k}={v}" for k, v in targets.items())
        ),
        ParameterBinding(key="trainDataset",           value=train_filename),
        ParameterBinding(key="testDataset",            value=test_filename),
        ParameterBinding(key="maximize",               value="true"),
        ParameterBinding(key="correctnessCutoff",      value="none"),
        ParameterBinding(key="includeFewShotExamples", value="false"),
        ParameterBinding(key="promptTemplateScope",    value="tenant"),
        ParameterBinding(key="prototypeMode",          value="false"),
        ParameterBinding(key="modelParams",            value="none"),
    ]

    # ── Mutually exclusive: the API allows only ONE of
    #    OPTIMIZATION_METRIC / FIELD_EVALUATION_METRICS / CUSTOM_METRIC_ID.
    if metric == "custom":
        input_parameters.append(ParameterBinding(key="customMetricId", value=CUSTOM_METRIC_ID))
    else:
        input_parameters.append(ParameterBinding(key="optimizationMetric", value=metric))

    input_artifacts = [
        InputArtifactBinding(key="prompt-data", artifact_id=prompt_artifact_id)
    ]

    params_dict = {p.key: p.value for p in input_parameters}

    try:
        existing = client.ai_core_client.configuration.query(
            scenario_id=SCENARIO, resource_group=resource_group
        )
        for conf in existing.resources:
            if {p.key: p.value for p in conf.parameter_bindings} == params_dict:
                print(f"Reusing configuration: {conf.id}")
                return conf.id
    except Exception as e:
        print(f"Could not query configs: {e}")

    resp = client.ai_core_client.configuration.create(
        name="bfcl-tool-config-custom",
        scenario_id=SCENARIO,
        executable_id=SCENARIO,
        resource_group=resource_group,
        parameter_bindings=input_parameters,
        input_artifact_bindings=input_artifacts,
    )
    print(f"Created configuration: {resp.id}")
    return resp.id

configuration_id = create_config(
    metric="custom",
    reference_model=REFERENCE_MODEL,
    targets=TARGET_MODELS,
    train_filename="bfcl_train.json",
    test_filename="bfcl_test.json",
    prompt_artifact_id=optimizer_artifact_id,
    prompt_name=PROMPT_NAME,
    prompt_version=PROMPT_VERSION,
    scenario=SCENARIO,
)
print(f"Configuration ID: {configuration_id}")

# ── Trigger execution ──────────────────────────────────────────────────────────
execution = client.ai_core_client.execution.create(
    configuration_id=configuration_id,
    resource_group=resource_group,
)
execution_id = execution.id
print(f"Execution ID: {execution_id}")

# ── Monitor ────────────────────────────────────────────────────────────────────
TERMINAL_STATES = {"COMPLETED", "FAILED", "DEAD", "STOPPED"}

while True:
    status = client.ai_core_client.execution.get(
        execution_id=execution_id, resource_group=resource_group
    )
    raw        = status.status.value if hasattr(status.status, "value") else str(status.status)
    status_str = raw.strip().upper()

    print(f"[{time.strftime('%H:%M:%S')}] {status_str}", end="")

    if hasattr(status, "status_details") and status.status_details:
        progress = status.status_details.get("progress", "")
        print(f"  progress={progress}", end="")
    print()

    if status_str in TERMINAL_STATES:
        if status_str != "COMPLETED":
            try:
                logs = client.ai_core_client.execution.get_logs(
                    execution_id=execution_id, resource_group=resource_group
                )
                print("── Execution logs ──")
                for log in logs.data:
                    print(log.msg)
            except Exception as e:
                print(f"Could not fetch logs: {e}")
        break

    time.sleep(30)

print(f"\nFinal status: {status_str}")

Reusing configuration: c47d2776-b93c-433e-b4ee-f0a99dd6b89b
Configuration ID: c47d2776-b93c-433e-b4ee-f0a99dd6b89b
Execution ID: e050725c62394035
[23:17:59] UNKNOWN
[23:18:30] RUNNING  progress=0/100
[23:19:01] RUNNING  progress=0/100
[23:19:32] RUNNING  progress=0/100
[23:20:03] RUNNING  progress=0/100
[23:20:33] RUNNING  progress=0/100
[23:21:05] RUNNING  progress=0/100
[23:21:35] RUNNING  progress=0/100
[23:22:06] RUNNING  progress=1/100
[23:22:37] RUNNING  progress=1/100
[23:23:08] RUNNING  progress=1/100
[23:23:39] RUNNING  progress=1/100
[23:24:11] RUNNING  progress=1/100
[23:24:42] RUNNING  progress=1/100
[23:25:13] RUNNING  progress=1/100
[23:25:44] RUNNING  progress=1/100
[23:26:15] RUNNING  progress=1/100
[23:26:46] RUNNING  progress=1/100
[23:27:17] RUNNING  progress=1/100
[23:27:47] RUNNING  progress=1/100
[23:28:18] RUNNING  progress=29/100
[23:28:49] RUNNING  progress=29/100
[23:29:21] RUNNING  progress=29/100
[23:29:52] RUNNING  progress=29/100
[23:30:23] RUNNING  progre

**Reading the output:** first you get a `Configuration ID` and `Execution ID`. Then each subsequent line is one status poll (roughly every 30 seconds) — notice the progress percentage jumping in stages (`0` → `1` → `31` → `80` → `100`) rather than climbing smoothly; this reflects discrete phases inside the optimizer's internal pipeline rather than a continuous progress bar. `Final status: COMPLETED` means a new, refined prompt has been written back to the Prompt Registry — that's what Step 8 retrieves next. If you instead see `FAILED`/`DEAD`/`STOPPED`, the cell auto-prints execution logs to help you diagnose the issue.

---

## Step 8 — Fetch the Full Optimized Prompt by ID

**What this step does:**
Retrieves the complete optimized prompt template from the Prompt Registry by its ID.

**Two-step lookup:** the Prompt Registry doesn't offer a "get by name" shortcut in this flow, so we first **list every prompt template** in the tenant (next cell) to find the `id` that corresponds to the name you're looking for (e.g. your own `targetPromptMapping` value from Step 6), and then **fetch that specific ID** (the cell after) to see its full content.

**Replace `optimized_id`** with the actual ID found in the listing below — the ID shown in this notebook is from a specific past run and won't match your own tenant.

The optimized prompt will be significantly more detailed than the base `"You are a helpful assistant."` — it will contain:
- A structured-output parser role definition
- Strict JSON output constraints (no markdown, no backticks)
- Full tool schemas across all 10 functions, with explicit normalization/derivation rules per field
- A conflict-resolution policy for handling repeated mentions of the same intent
- Internal (non-revealed) reasoning steps and output-formatting rules

In [17]:
url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
res = requests.get(url, headers=client.request_header)
templates = res.json()
for t in templates.get("resources", []):
    print(f"  name={t['name']}  version={t['version']}  id={t['id']}")


  name=multi_task_withRG  version=1.1.3  id=35f5c235-b949-49bb-854e-cca0913086ab
  name=expand_text  version=1.1.2  id=c050ab3a-1653-41d5-8d76-1c2f0a4457fd
  name=multi_task_withRG  version=1.1.2  id=9b502632-8ddc-4d44-bc0c-78044f8ca63b
  name=multi_task  version=1.1.1  id=fca6185e-e340-4781-9d13-00e32f510674
  name=facility-json-template  version=1.0.0  id=27ac3122-9b6a-4baa-a7a2-c3670cea83b2
  name=prompt-registry-eval-demo  version=1.0.0  id=9f47e745-5c87-46df-b917-8e7dc829f47c
  name=evalPromptTemplateConfig-227e9e3  version=1.0.0  id=d4416a6f-8f45-458e-81a2-f3d2f346a26e
  name=evalPromptTemplateConfig-8eb5f38  version=1.0.0  id=87faced1-57b6-4445-aa3f-6294f1a5a8b0
  name=evalPromptTemplateConfig-25152b4  version=1.0.0  id=24dfbaad-9609-4a69-abf9-e3053aa90bf2
  name=evalPromptTemplateConfig-de19a80  version=1.0.0  id=5ac8367e-fc1d-4aed-87ae-cb5166112b1c
  name=evalPromptTemplateConfig-fa00f03  version=1.0.0  id=ef294c8f-ded7-4ddf-a3ec-aa415efc7300
  name=evalPromptTemplateConfig-f5

**Reading the output:** this is a directory listing of every prompt template in your tenant — scan it for the name you registered as the optimizer's output target (set in `targetPromptMapping` in Step 6), and copy its `id` value into the next cell. In this run, `bfcl-tool-optimized-custom` at version `0.0.1` maps to `id=cfb19fdf-b070-42b6-8a78-3bde08b0c4af` — and that's exactly the ID used in the next cell, so this is a good example of the listing → fetch pattern working end-to-end.

In [18]:
# ── Fetch optimized prompt template ──────────────────────────────────────────
# Replace with the actual ID of bfcl-tool-optimized-gemini25 from the listing above
optimized_id = "cfb19fdf-b070-42b6-8a78-3bde08b0c4af"

url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{optimized_id}"
res = requests.get(url, headers=client.request_header)
print(f"Status: {res.status_code}")
optimized = res.json()
print(json.dumps(optimized, indent=2))

Status: 200
{
  "id": "cfb19fdf-b070-42b6-8a78-3bde08b0c4af",
  "name": "bfcl-tool-optimized-custom",
  "version": "0.0.1",
  "scenario": "genai-optimizations",
  "creationTimestamp": "2026-06-30T06:26:48.863000",
  "managedBy": "imperative",
  "isVersionHead": false,
  "spec": {
    "template": [
      {
        "role": "system",
        "content": "You are a structured-output extraction assistant. Your sole task is to read a user query and return a single top-level JSON object that strictly conforms to the provided schema. Output JSON only \u2014 no prose, no headings, no markdown, no code fences, no backticks, no explanations, and no placeholders. Begin the response with { and end with }. If nothing is extractable, return {}."
      },
      {
        "role": "user",
        "content": "# Role and Objective\nExtract all applicable intents from the user\u2019s query and emit a single top-level JSON object that strictly follows the schema below. Do not include any narrative text in yo

**Reading the output:** the fetched `spec.template` shows two messages — `system` and `user`. Notice how much more detailed this is than the two-line base prompt: it defines 10 supported "intents" (one per tool) with explicit field types and normalization rules, a conflict-resolution policy for repeated mentions of the same intent, and strict output-formatting instructions (`Begin the response with { and end with }`, no markdown, no trailing commas). This level of specificity is exactly what the optimizer generated automatically by iterating against your `tool-call-accuracy` custom metric — you didn't write any of this by hand.

---

## Step 09 — Compare Base vs Optimized Prompt via Live Inference

**What this step does:**
Runs four test questions through both the base and optimized prompts on Gemini 2.5 Pro and compares the outputs side by side. This is the "proof in the pudding" step — everything before this was setup and orchestration; this is where you actually *see* whether the optimization helped.

**For each question the comparison shows:**
- `BASE` output — what the base prompt (`"You are a helpful assistant."`) produces
- `OPTIMIZED` output — what the optimizer-refined prompt produces
- `COMPARISON` — whether each output is valid JSON and which tools were called
- `VERDICT` — whether the optimization was a WIN, both valid, or both invalid

**Expected result:**
The base prompt typically returns natural language prose. The optimized prompt returns a strict JSON object of tool calls — confirming a successful optimization WIN.

```
BASE         → invalid JSON ❌ | raw: Of course! I can help with all three...
OPTIMIZED    → valid JSON ✅ | tools called: ['currency_conversion', 'event_finder', 'recipe_search']

📈 VERDICT:
🏆 Optimization WIN — base gave prose, optimized gave structured JSON
```

### 9.1 — Finding a live orchestration deployment

Live inference in AI Core runs through a **deployed** orchestration scenario (a running service with its own URL), not just an API key + model name. This cell lists every deployment in your tenant so you can find one with `scenario=orchestration` and `status=RUNNING`, then copies its URL into `ORCHESTRATION_DEPLOYMENT_URL` in the next cell. If you don't have a running orchestration deployment yet, you'd need to create one first via the AI Core console/API before this step will work.

In [19]:
# ── Step 9: Compare Base vs Optimized Prompt via Live Inference ───────────────
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.models.template import Template
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService

# Find the running orchestration deployment URL
url = f"{client.ai_core_client.base_url}/lm/deployments"
res = requests.get(url, headers=client.request_header)
for d in res.json().get("resources", []):
    print(f"id={d.get('id')}  scenario={d.get('scenarioId'):30s}  status={d.get('status'):10s}  url={d.get('deploymentUrl')}")

id=ddee9146ce463a8b  scenario=orchestration                   status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/ddee9146ce463a8b
id=d46fa82110bbc6c8  scenario=foundation-models               status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d46fa82110bbc6c8
id=d6fa93ca356f105a  scenario=foundation-models               status=STOPPED     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d6fa93ca356f105a
id=d0d13204b3862077  scenario=foundation-models               status=RUNNING     url=https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d0d13204b3862077
id=d0048ce66805cd7a  scenario=foundation-models               status=RUNNING     url=wss://realtime.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/d0048ce66805cd7a
id=dc0c40f766de1b52  scenario=or

### 9.2 — Running the comparison

This cell does several things in sequence:

1. **Fetch both prompt templates** — loads the base and optimized templates by ID/name from the registry (same lookup pattern as Step 8) and splits each into its `system` and `user` message content.
2. **`run_inference`** — a small wrapper around `OrchestrationService` that substitutes the question into the user template, builds an `OrchestrationConfig` (model + system/user messages), and runs it against the deployment URL found above, returning the model's raw text response.
3. **`clean_json_output`** — strips markdown code fences (```` ``` ````) that some models wrap around JSON output before parsing, so a technically-correct JSON response wrapped in a code fence doesn't get incorrectly flagged as invalid.
4. **`compare_prompts`** — runs a single question through *both* prompts, attempts `json.loads()` on each cleaned output, prints whether each is valid JSON and which tool names were found (the top-level keys), and produces a final verdict:
   - 🏆 **WIN** — base produced invalid JSON/prose, optimized produced valid JSON (the ideal outcome, proving the optimized prompt reliably constrains the model to structured output)
   - ✅ **Both valid** — compare tool names/accuracy manually, since both technically "worked"
   - ⚠️ / ❌ — worth investigating; could indicate a flaky model response or a deployment issue rather than a genuine regression
5. **Run all comparisons** — calls `compare_prompts` on four different multi-tool questions covering different domains (weather, currency conversion + restaurant + stock, hotel booking + weather + distance, currency + events + recipes) to get a broader sense of how consistently the optimized prompt outperforms the base.

**Reading the output:** for the first question ("What is the weather in Tokyo...") the base prompt apologizes that it has no real-time data access and suggests external weather sites — completely unusable as a tool call. The optimized prompt instead returns a clean `weather_forecast` JSON object with normalized fields (`location`, `days`, `units`) — exactly the structured output a downstream system could act on. This pattern repeats across all four test questions, confirming the optimization reliably improved structured-output reliability for this target model.

In [21]:


# ── Fetch both prompt templates from the registry ──────────────────────────────
def get_prompt_template(template_id: str) -> dict:
    url = f"{client.ai_core_client.base_url}/lm/promptTemplates/{template_id}"
    res = requests.get(url, headers=client.request_header)
    res.raise_for_status()
    return res.json()

def extract_messages(template: dict) -> dict:
    messages = {}
    for msg in template.get("spec", {}).get("template", []):
        messages[msg["role"]] = msg["content"]
    return messages

url = f"{client.ai_core_client.base_url}/lm/promptTemplates"
res = requests.get(url, headers=client.request_header)
all_templates = {
    f"{t['name']}:{t['version']}": t["id"]
    for t in res.json().get("resources", [])
}

base_id      = all_templates.get(f"{PROMPT_NAME}:{PROMPT_VERSION}")
optimized_id = all_templates.get(list(TARGET_MODELS.values())[0])

base_template      = get_prompt_template(base_id)
optimized_template = get_prompt_template(optimized_id)

base_messages      = extract_messages(base_template)
optimized_messages = extract_messages(optimized_template)

print("✅ Loaded base and optimized prompt templates.")
print(f"Base system prompt      : {base_messages['system'][:80]}...")
print(f"Optimized system prompt : {optimized_messages['system'][:80]}...")

# ── Paste your running orchestration deployment URL here ──────────────────────
ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.internalprod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/ddee9146ce463a8b"

def run_inference(system_prompt: str, user_template: str, question: str, model_name: str) -> str:
    user_content = user_template.replace("{{?question}}", question)

    config = OrchestrationConfig(
        llm=LLM(name=model_name),
        template=Template(messages=[
            SystemMessage(system_prompt),
            UserMessage(user_content),
        ]),
    )

    service  = OrchestrationService(api_url=ORCHESTRATION_DEPLOYMENT_URL, config=config)
    response = service.run()
    return response.module_results.llm.choices[0].message.content


def clean_json_output(text: str) -> str:
    """Strip markdown code fences that models sometimes wrap around JSON."""
    text = text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        lines = [l for l in lines if not l.strip().startswith("```")]
        text = "\n".join(lines).strip()
    return text


def compare_prompts(question: str, model_name: str = "gemini-2.5-pro"):
    print("\n" + "=" * 70)
    print(f"QUESTION:\n{question}")
    print("=" * 70)

    print("\n📌 BASE PROMPT OUTPUT:")
    print("-" * 40)
    try:
        base_output = run_inference(
            system_prompt=base_messages["system"],
            user_template=base_messages["user"],
            question=question,
            model_name=model_name,
        )
        print(base_output)
    except Exception as e:
        base_output = f"ERROR: {e}"
        print(base_output)

    print("\n✅ OPTIMIZED PROMPT OUTPUT:")
    print("-" * 40)
    try:
        optimized_output = run_inference(
            system_prompt=optimized_messages["system"],
            user_template=optimized_messages["user"],
            question=question,
            model_name=model_name,
        )
        print(optimized_output)
    except Exception as e:
        optimized_output = f"ERROR: {e}"
        print(optimized_output)

    print("\n📊 COMPARISON:")
    print("-" * 40)
    for label, output in [("BASE", base_output), ("OPTIMIZED", optimized_output)]:
        cleaned = clean_json_output(output)
        try:
            parsed = json.loads(cleaned)
            tools  = list(parsed.keys())
            print(f"{label:12s} → valid JSON ✅ | tools called: {tools}")
        except json.JSONDecodeError:
            print(f"{label:12s} → invalid JSON ❌ | raw: {output[:120]}")

    print("\n📈 VERDICT:")
    print("-" * 40)
    base_valid      = True
    optimized_valid = True
    try:
        json.loads(clean_json_output(base_output))
    except Exception:
        base_valid = False
    try:
        json.loads(clean_json_output(optimized_output))
    except Exception:
        optimized_valid = False

    if not base_valid and optimized_valid:
        print("🏆 Optimization WIN — base gave prose, optimized gave structured JSON")
    elif base_valid and optimized_valid:
        print("✅ Both valid JSON — compare tool accuracy above")
    elif base_valid and not optimized_valid:
        print("⚠️  Base was valid but optimized was not — check prompt")
    else:
        print("❌ Both invalid — check model or deployment")

    return base_output, optimized_output


# ── Run all comparisons ─────────────────────────────────────────────────────────
compare_prompts("What is the weather in Tokyo for the next 3 days in celsius?")

compare_prompts(
    "I have 2000 euros and want to know how much that is in USD. "
    "Also find me a mid-range Italian restaurant in Milan. "
    "And what is Apple's current stock price?"
)

compare_prompts(
    "Book a hotel in Paris for 2 guests from 2025-08-01 to 2025-08-05 "
    "in a deluxe room. Also check the weather in Paris for the next 7 days "
    "in celsius. And find me the distance from Paris to Lyon in km."
)

compare_prompts(
    "Convert 5000 US dollars to Japanese yen. "
    "Find concerts in New York tomorrow. "
    "Search for vegan pasta recipes under 30 minutes."
)

✅ Loaded base and optimized prompt templates.
Base system prompt      : You are a helpful assistant....
Optimized system prompt : You are a structured-output intent and argument parser. Your sole task is to ext...

QUESTION:
What is the weather in Tokyo for the next 3 days in celsius?

📌 BASE PROMPT OUTPUT:
----------------------------------------
As an AI, I don't have access to live, real-time weather data. My knowledge has a cutoff point, and I cannot provide an up-to-the-minute forecast.

However, I can give you the best ways to get a reliable forecast and show you what a typical one looks like.

### **Where to Get a Live Forecast for Tokyo:**

For the most accurate and current information, please use one of these sources:

1.  **Google Search:** Simply search for "**weather in Tokyo**". Google provides a detailed, live 3-day forecast at the top of the search results.
2.  **Reliable Weather Websites:**
    *   **Japan Meteorological Agency (JMA):** The official source for weather i

('Of course! Here is the information you requested, broken down by each task:\n\n### 1. USD to JPY Conversion\n\nBased on current mid-market exchange rates, 5000 US dollars is approximately:\n\n**¥785,500 Japanese Yen**\n\n**Important:** This is an estimate. Exchange rates fluctuate constantly. The actual rate you receive from a bank, credit card, or currency exchange service will be slightly different due to transaction fees and their specific buy/sell rate. Always check with your financial institution for the exact rate before making a transaction.\n\n---\n\n### 2. Concerts in New York Tomorrow\n\nFinding a complete, real-time list is best done through dedicated services, as schedules can change. However, here are some of the best resources to find out who is playing in NYC tomorrow:\n\n*   **Check these websites:**\n    *   **Songkick:** ([https://www.songkick.com/metro_areas/28399-us-new-york](https://www.songkick.com/metro_areas/28399-us-new-york)) - Excellent for tracking your fa

---

## 🔍 Bonus Analysis — How Does the Custom-Metric-Optimized Prompt Compare to a `JSON_Match`-Optimized One?

Since this notebook creates its *own* custom metric, it's natural to ask: does using `tool-call-accuracy` (LLM-as-a-judge) instead of the built-in `JSON_Match` actually change what the optimizer produces? The note below reflects a direct comparison between the two resulting prompts (each optimized separately, one per metric, on the same base prompt and dataset):

The custom metric output is consistently more verbose/detailed in its location and entity normalization — it tends to include both the city and the country/state qualifier ("Tokyo, Japan" vs "Tokyo"), and it more reliably includes optional fields like max_prep_time that JSON_Match dropped entirely in the last test case.
JSON_Match output is more minimal — single values per field, fewer optional fields populated, and the structure feels closer to "just enough to match a golden answer" rather than "as complete as possible."

**Why this makes sense:** `JSON_Match` rewards prompts whose output structurally matches the golden answer as closely as possible — which tends to push the optimizer toward *minimal, exact* outputs, since anything extra risks a mismatch. The LLM-as-a-judge `tool-call-accuracy` metric instead rewards outputs that are judged *complete and correct* by a rubric, which gives the optimizer more freedom to be thorough (e.g. including both a canonical location name and a common alias, or populating optional fields like `max_prep_time` when they're genuinely inferable) without being penalized for not matching the golden answer character-for-character.

**Takeaway:** neither approach is strictly "better" in the abstract — it depends on what your downstream system needs. If your tool-calling consumer expects a tight, minimal argument set, `JSON_Match` may produce a more predictable prompt. If it can handle (or benefits from) richer, more complete extractions, an LLM-as-a-judge custom metric like `tool-call-accuracy` may serve you better.

---

## Summary

In this notebook you completed the following steps:

1. ✅ **Connected to AI Core** — loaded credentials and initialized `GenAIHubProxyClient`
2. ✅ **Verified and created** the custom LLM-as-a-judge metric `tool-call-accuracy:1.0.0` directly via the API
3. ✅ **Configured parameters** — dataset path, prompt name, models, and metric
4. ✅ **Loaded and normalized** the BFCL v3 dataset — robust multi-format reader, tool normalization, golden record builder
5. ✅ **Uploaded 4 files** to AI Core dataset storage — train, test, tools, prompt template
6. ✅ **Registered a dataset artifact** — shared folder linked to `genai-optimizations` scenario
7. ✅ **Pushed base prompt template** to the Prompt Registry — `bfcl-tool-base:0.0.1`
8. ✅ **Created a configuration, triggered, and monitored** the optimization execution in a single combined step — polled until `COMPLETED`
9. ✅ **Retrieved the optimized prompt** — `bfcl-tool-optimized-custom:0.0.1` from the registry
10. ✅ **Compared base vs optimized** via live inference — confirmed optimization WIN across all four test questions
11. ✅ **Compared the custom-metric prompt against a `JSON_Match`-optimized prompt** — observed the custom metric yields more complete, verbose extractions

## 🧩 Key concepts recap

- **Artifacts** turn raw files into referenceable inputs for AI Core jobs.
- **Configurations** are reusable "recipes"; **executions** are individual runs of a recipe.
- **Custom LLM-as-a-judge metrics** let you evaluate prompt quality on nuanced, rubric-based criteria instead of a rigid exact-match — and, as shown here, can be created programmatically instead of only through Bruno/Postman.
- The **train/test split** exists to make sure the reported optimization score reflects genuine generalization, not memorization.
- The final measure of success isn't the optimizer's internal score alone — it's whether the optimized prompt actually behaves better in **live inference**, which is why Step 9 exists.
- **The choice of metric shapes the optimizer's output** — an exact-match metric like `JSON_Match` tends to produce minimal, tightly-scoped prompts, while an LLM-as-a-judge metric tends to produce more thorough, complete extractions.

## 🚀 Possible next steps

- **Define `create_config` at the top of this notebook** (see the callout in Step 6/7) so this notebook can run standalone without depending on a prior kernel session.
- **Swap in your own dataset** — replace the BFCL file with your own labeled question/tool-call pairs (matching the golden record format from Step 3) to optimize prompts for your own tools.
- **Try a different target model** — add more entries to `TARGET_MODELS` in Step 2 to optimize the same base prompt for multiple models in one configuration.
- **Tighten the custom metric's rubric** — revisit the `ratingRubric` / `evaluationSteps` in the metric-creation cell and add more specific scoring criteria if you notice systematic gaps in Step 9's results.
- **Promote to production** — once you're satisfied with the optimized prompt, reference it by name+version directly in your application's orchestration calls instead of hard-coding prompt text.
- **Build a small internal tutorial/app** around this flow — e.g. a "prompt optimization playground" that lets a user upload their own dataset, pick a metric (built-in or custom), and get back a comparison report like Step 9's, without touching notebook code directly.

## 🛠️ Troubleshooting quick-reference

| Symptom | Likely cause |
|---|---|
| `NameError: name 'create_config' is not defined` (Step 6/7 cell) | See the callout in that section — paste in the missing function definition |
| `❌ Custom metric 'tool-call-accuracy' not found`, and the create cell also fails | Check the `scenario` value matches (`genai-optimizations`) and that your resource group has permission to create evaluation metrics |
| Execution status goes to `FAILED`/`DEAD` | Check the auto-printed execution logs first; common causes are an unavailable model in your region, a malformed dataset file, or an invalid `CUSTOM_METRIC_ID` |
| Can't find your optimized prompt by name in Step 8's listing | Double check the exact name you set in `targetPromptMapping` (Step 6) — it must match exactly, including version |
| No `RUNNING` orchestration deployment found (Step 9) | You need an active orchestration deployment in AI Core before live inference comparisons will work — create one via the console/API if none exists |
| `compare_prompts` shows both outputs as invalid JSON | Could indicate a transient model issue — try re-running, or verify the deployment URL is still `RUNNING` |

